# 🔮 Guanacaste Tourism Forecaster — Fase 4: Modelado Predictivo
### Cascade Forecasting con Facebook Prophet (2026-2027)
---
**Estrategia:** Predicción en 2 etapas:
1. **Stage 1 — Arrivals:** Proyectar llegadas SJO y LIR hacia 2027
2. **Stage 2 — Occupancy:** Usar esas proyecciones como regresores para predecir la ocupación hotelera

**Variables de impacto incluidas:** Semana Santa (2009-2027), COVID-19, Crisis Narco 2023, Colapso Pista LIR 2024

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Helper de visualización (sin dependencia de nbformat)
def show(fig, height=500):
    html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    display(HTML(f'<div style="height:{height}px">{html}</div>'))

TEMPLATE = 'plotly_dark'
print('✅ Entorno de Modelado Listo.')

### 1. Carga del Dataset Maestro

In [ ]:
df = pd.read_csv('../data/merged/master_tourism_data.csv', parse_dates=['DATE'])
print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Rango temporal: {df["DATE"].min().date()} → {df["DATE"].max().date()}')
display(df.head(3))

### 2. Definición de Eventos y Festivos (Semana Santa + Shocks)
Incluimos la Semana Santa para todos los años 2009-2027, ya que es el pico turístico principal en Guanacaste.

In [ ]:
# Fechas exactas de Semana Santa (Viernes Santo) — pico local
semana_santa_dates = [
    '2009-04-10', '2010-04-02', '2011-04-22', '2012-04-06',
    '2013-03-29', '2014-04-18', '2015-04-03', '2016-03-25',
    '2017-04-14', '2018-03-30', '2019-04-19', '2020-04-10',
    '2021-04-02', '2022-04-15', '2023-04-07', '2024-03-29',
    '2025-04-18', '2026-04-03', '2027-03-26'
]

holidays_df = pd.DataFrame({
    'holiday': 'semana_santa',
    'ds': pd.to_datetime(semana_santa_dates),
    'lower_window': -7,   # Una semana antes comienza el pico
    'upper_window': 2     # 2 días después
})

# Agregar shocks negativos
shocks = pd.DataFrame([
    {'holiday': 'covid_lockdown',    'ds': '2020-03-15', 'lower_window': 0, 'upper_window': 640},
    {'holiday': 'crisis_narco',      'ds': '2023-01-01', 'lower_window': 0, 'upper_window': 365},
    {'holiday': 'colapso_pista_lir', 'ds': '2024-11-01', 'lower_window': 0, 'upper_window': 45},
])
shocks['ds'] = pd.to_datetime(shocks['ds'])
holidays_df = pd.concat([holidays_df, shocks], ignore_index=True)

print(f'✅ {len(holidays_df)} eventos definidos (festivos + shocks).')

---
## 🛫 Stage 1: Proyección de Llegadas (SJO & LIR)
Primero predecimos cuántos viajeros llegarán hasta 2027, para usarlos como insumo en la predicción de ocupación.

In [ ]:
def train_arrivals_model(df, col, name):
    train = df[['DATE', col]].dropna().rename(columns={'DATE': 'ds', col: 'y'})

    model = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        holidays=holidays_df,
        changepoint_prior_scale=0.05
    )
    model.fit(train)
    print(f'✅ Modelo {name} entrenado con {len(train)} meses de historia.')
    return model

model_sjo = train_arrivals_model(df, 'Arrivals_sjo', 'SJO')
model_lir = train_arrivals_model(df, 'Arrivals_lir', 'LIR')

In [ ]:
# Proyectar 24 meses hacia adelante (hasta finales de 2027)
future_sjo = model_sjo.make_future_dataframe(periods=24, freq='MS')
future_lir = model_lir.make_future_dataframe(periods=24, freq='MS')

fc_sjo = model_sjo.predict(future_sjo)
fc_lir = model_lir.predict(future_lir)

# Unir proyecciones
df_future = fc_sjo[['ds', 'yhat']].rename(columns={'yhat': 'Arrivals_sjo_fc'})
df_future = df_future.merge(fc_lir[['ds', 'yhat']].rename(columns={'yhat': 'Arrivals_lir_fc'}), on='ds')
df_future['Arrivals_sjo_fc'] = df_future['Arrivals_sjo_fc'].clip(lower=0)
df_future['Arrivals_lir_fc'] = df_future['Arrivals_lir_fc'].clip(lower=0)

print(f'✅ Proyección de llegadas generada hasta {df_future["ds"].max().date()}')

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Proyección SJO — Alajuela', 'Proyección LIR — Guanacaste'),
    vertical_spacing=0.12)

# SJO
hist_sjo = df[['DATE', 'Arrivals_sjo']].dropna()
fig.add_trace(go.Scatter(x=hist_sjo['DATE'], y=hist_sjo['Arrivals_sjo'],
    name='SJO Histórico', line=dict(color='#00d4ff', width=2)), row=1, col=1)
future_only = df_future[df_future['ds'] > hist_sjo['DATE'].max()]
fig.add_trace(go.Scatter(x=df_future['ds'], y=df_future['Arrivals_sjo_fc'],
    name='SJO Proyectado', line=dict(color='#00d4ff', width=2, dash='dash')), row=1, col=1)
fig.add_vrect(x0=str(hist_sjo['DATE'].max()), x1=str(df_future['ds'].max()),
    fillcolor='rgba(255,255,255,0.03)', line_width=0, row=1, col=1)

# LIR
hist_lir = df[['DATE', 'Arrivals_lir']].dropna()
fig.add_trace(go.Scatter(x=hist_lir['DATE'], y=hist_lir['Arrivals_lir'],
    name='LIR Histórico', line=dict(color='#00ff9d', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_future['ds'], y=df_future['Arrivals_lir_fc'],
    name='LIR Proyectado', line=dict(color='#00ff9d', width=2, dash='dash')), row=2, col=1)
fig.add_vrect(x0=str(hist_lir['DATE'].max()), x1=str(df_future['ds'].max()),
    fillcolor='rgba(255,255,255,0.03)', line_width=0, row=2, col=1)

fig.update_layout(title='<b>Stage 1: Proyección de Llegadas Internacionales 2026-2027</b>',
    template=TEMPLATE, height=600, hovermode='x unified',
    legend=dict(orientation='h', y=-0.12))
show(fig, height=620)

---
## 🏨 Stage 2: Proyección de Ocupación Hotelera en Guanacaste
Usamos las llegadas proyectadas (SJO + LIR) como regresores adicionales para mejorar la precisión.

In [ ]:
# Preparar dataset de entrenamiento para ocupación
df_occ_train = df[['DATE', 'Guanacaste_Occupancy_Pct', 'Arrivals_sjo', 'Arrivals_lir']].dropna()
df_occ_train = df_occ_train.rename(columns={
    'DATE': 'ds',
    'Guanacaste_Occupancy_Pct': 'y',
    'Arrivals_sjo': 'arrivals_sjo',
    'Arrivals_lir': 'arrivals_lir'
})

model_occ = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    holidays=holidays_df,
    changepoint_prior_scale=0.05,
    interval_width=0.90
)
model_occ.add_regressor('arrivals_sjo', standardize=True)
model_occ.add_regressor('arrivals_lir', standardize=True)
model_occ.fit(df_occ_train)

print(f'✅ Modelo de Ocupación entrenado con {len(df_occ_train)} meses históricos.')

In [ ]:
# Construir dataframe futuro con llegadas proyectadas
df_future_occ = df_future.rename(columns={
    'ds': 'ds',
    'Arrivals_sjo_fc': 'arrivals_sjo',
    'Arrivals_lir_fc': 'arrivals_lir'
})

fc_occ = model_occ.predict(df_future_occ)
fc_occ['yhat'] = fc_occ['yhat'].clip(lower=0, upper=100)
fc_occ['yhat_lower'] = fc_occ['yhat_lower'].clip(lower=0, upper=100)
fc_occ['yhat_upper'] = fc_occ['yhat_upper'].clip(lower=0, upper=100)

print(f'✅ Proyección de Ocupación generada: {fc_occ["ds"].min().date()} → {fc_occ["ds"].max().date()}')

---
## 📈 Resultado Final: Proyección de Ocupación Hotelera Guanacaste 2026-2027

In [ ]:
hist_occ = df[['DATE', 'Guanacaste_Occupancy_Pct']].dropna()
cutoff = hist_occ['DATE'].max()
future_fc = fc_occ[fc_occ['ds'] > cutoff]

fig = go.Figure()

# Banda de incertidumbre (90%)
fig.add_trace(go.Scatter(
    x=pd.concat([future_fc['ds'], future_fc['ds'][::-1]]),
    y=pd.concat([future_fc['yhat_upper'], future_fc['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(255,107,53,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Intervalo 90% confianza',
    hoverinfo='skip'
))

# Histórico
fig.add_trace(go.Scatter(
    x=hist_occ['DATE'], y=hist_occ['Guanacaste_Occupancy_Pct'],
    name='Ocupación Histórica', line=dict(color='#ffffff', width=2)
))

# Proyección
fig.add_trace(go.Scatter(
    x=future_fc['ds'], y=future_fc['yhat'],
    name='Proyección 2026-2027', line=dict(color='#ff6b35', width=3, dash='dash')
))

# Línea de corte historico/futuro
fig.add_vline(x=cutoff, line_dash='dot', line_color='rgba(255,255,255,0.4)',
    annotation_text='Hoy', annotation_position='top right')

fig.update_layout(
    title='<b>🏨 Proyección de Ocupación Hotelera — Guanacaste 2026-2027</b>',
    xaxis_title='Fecha',
    yaxis_title='Ocupación (%)',
    template=TEMPLATE,
    height=500,
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.15),
    font=dict(family='Arial, sans-serif')
)
show(fig, height=520)

In [ ]:
# Exportar el forecast final
output = fc_occ[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
output.columns = ['Fecha', 'Ocupacion_Proyectada_Pct', 'Limite_Inferior_90pct', 'Limite_Superior_90pct']
output = output[output['Fecha'] > cutoff]

os.makedirs('../reports', exist_ok=True)
output.to_csv('../reports/forecast_ocupacion_2026_2027.csv', index=False)

print('✅ Forecast exportado a reports/forecast_ocupacion_2026_2027.csv')
print('\n📋 Resumen del Forecast:')
display(output.set_index('Fecha').round(1))

In [ ]:
# Descomposición de componentes del modelo
from prophet.plot import plot_components_plotly

comp_fig = plot_components_plotly(model_occ, fc_occ)
comp_fig.update_layout(template=TEMPLATE, title='<b>Componentes del Modelo (Tendencia + Estacionalidad + Eventos)</b>')
show(comp_fig, height=700)